In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# מבוא לפרימיטיבים

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## מדוע Qiskit הציגה פרימיטיבים?
בדומה לימים הראשונים של המחשבים הקלאסיים, כאשר מפתחים היו צריכים לתפעל רגיסטרים של ה-CPU ישירות, הממשק המוקדם ל-QPUs החזיר פשוט את הנתונים הגולמיים מהאלקטרוניקה השולטת.
זה לא היה בעיה גדולה כאשר QPUs שכנו במעבדות והרשו גישה ישירה רק לחוקרים.
מתוך הכרה בכך שרוב המפתחים לא יכירו ולא צריכים להכיר את אופן הפענוח של נתונים גולמיים כאלה ל-0 ו-1, Qiskit הציגה את `backend.run`, הפשטה ראשונה לגישה ל-QPUs בענן. זה אפשר למפתחים לעבוד עם פורמט נתונים מוכר ולהתמקד בתמונה הגדולה.

כאשר הגישה ל-QPUs הפכה נפוצה יותר, ועם יותר אלגוריתמים קוונטיים שפותחו, עלתה שוב הצורך בהפשטה ברמה גבוהה יותר. בתגובה, Qiskit הציגה את ממשק הפרימיטיבים, המותאם לשתי משימות ליבה בפיתוח אלגוריתמים קוונטיים: אמדון ערך ציפייה (`Estimator`) ודגימת Circuit (`Sampler`). המטרה היא שוב לעזור למפתחים להתמקד יותר בחדשנות ופחות בהמרת נתונים. ממשק הפרימיטיבים מחליף את ממשק `backend.run`, שכן `Sampler` מספק את אותה גישה ישירה לחומרה שהוצעה על ידי `backend.run`.

## מהו פרימיטיב?
מערכות מחשוב בנויות על שכבות מרובות של הפשטה. הפשטות מאפשרות לך להתמקד ברמת פרטים מסוימת הרלוונטית למשימה שבידיך. ככל שמתקרבים לחומרה, כך נמוכה יותר רמת ההפשטה הנדרשת (לדוגמה, ייתכן שתצטרך להזיז או לתפעל נתונים ברמת הוראות ה-CPU). ככל שהמשימה שברצונך לבצע מורכבת יותר, כך ההפשטות יהיו ברמה גבוהה יותר (לדוגמה, אפשר שתשתמש בספרייה לביצוע חישובים אלגבריים).

בהקשר זה, *פרימיטיב* הוא הוראת העיבוד הקטנה ביותר, אבן הבניין הפשוטה ביותר ממנה ניתן ליצור משהו שימושי לרמת הפשטה נתונה.

ההתקדמות האחרונה בחישוב קוונטי הגבירה את הצורך לעבוד ברמות גבוהות יותר של הפשטה. ככל שהתחום מתקדם לעבר יחידות עיבוד קוונטי (QPUs) גדולות יותר וזרימות עבודה מורכבות יותר, המיקוד עובר מאינטראקציה עם אותות qubit בודדים לראיית המכשירים הקוונטיים כמערכות המבצעות משימות הכרחיות.

שתי המשימות הנפוצות ביותר למחשבים קוונטיים הן דגימת מצבים קוונטיים וחישוב ערכי ציפייה. משימות אלה הניעו את עיצוב פרימיטיבי Qiskit: **Estimator** ו-**Sampler**.

- Estimator מחשב ערכי ציפייה של אובזרבלים ביחס למצבים המוכנים על ידי Circuits קוונטיות.
- Sampler דוגם את רגיסטר הפלט מהרצת Circuit קוונטית.

בקצרה, מודל החישוב שהוצג על ידי פרימיטיבי Qiskit מקדם את התכנות הקוונטי צעד אחד קרוב יותר למקום שבו נמצא התכנות הקלאסי כיום, שבו המיקוד פחות על פרטי החומרה ויותר על התוצאות שאתה מנסה להשיג.

## הגדרת פרימיטיב ומימושים
ישנם שני סוגים של פרימיטיבי Qiskit: המחלקות הבסיסיות והמימושים שלהן. פרימיטיבי Estimator ו-Sampler מוגדרים על ידי מחלקות בסיס פרימיטיב בקוד פתוח שחיות ב-Qiskit SDK (במודול [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). ספקים (כגון Qiskit Runtime) יכולים להשתמש במחלקות בסיס אלה כדי לגזור מימושי Sampler ו-Estimator משלהם. רוב המשתמשים יתקשרו עם מימושי הספקים, לא עם הפרימיטיבים הבסיסיים.

### מחלקות בסיס
פרימיטיבי ה-`Base` הם מחלקות מופשטות המגדירות ממשק משותף למימוש פרימיטיבים. כל שאר המחלקות במודול [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) יורשות ממחלקות בסיס אלה. מפתחים צריכים להשתמש בהן אם הם מעוניינים ליצור מודל הרצה משלהם מבוסס פרימיטיבים עבור ספק ספציפי. מחלקות אלה עשויות להיות שימושיות גם למי שרוצה לבצע עיבוד מותאם מאוד ומגלה שמימושי הפרימיטיבים הקיימים פשוטים מדי לצרכיו. משתמשים כלליים לא ישתמשו ישירות במחלקות הבסיס.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) ו-[`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) — למרות שפרימיטיבי V1 עדיין ניתנים לשימוש, מדריכים אלה מתמקדים בפרימיטיבי V2 מאחר שהם העדכניים ביותר ונפוצים יותר.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) ו-[`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) — פרימיטיבי הייחוס של Qiskit עוקבים אחר מפרטי ממשק אלה.

<span id="implementations"></span>
### מימושים
כל הפרימיטיבים נוצרים ממחלקות הבסיס; לפיכך, יש להם אותה מבנה ושימוש כלליים. לדוגמה, פורמט הקלט עבור כל פרימיטיבי Estimator זהה. עם זאת, ישנם הבדלים במימושים שהופכים אותם לייחודיים.

אלה הם מימושים של מחלקות הבסיס של הפרימיטיבים:

- [פרימיטיבי Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) ו-[`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), מספקים מימוש מתוחכם יותר (לדוגמה, על ידי הכללת הפחתת שגיאות) כשירות מבוסס ענן. מימוש זה של הפרימיטיבים הבסיסיים משמש לגישה לחומרה של IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) ו-[`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) — מימושי ייחוס של הפרימיטיבים המשתמשים בסימולטור המובנה ב-Qiskit. הם בנויים עם מודול [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) של Qiskit, ומייצרים תוצאות המבוססות על סימולציות וקטור מצב אידיאליות. הם נגישים דרך Qiskit. ראה [סימולציה מדויקת עם פרימיטיבי Qiskit](/guides/simulate-with-qiskit-sdk-primitives) לפרטי שימוש.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) ו-[`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) — ניתן להשתמש במחלקות אלה כדי "לעטוף" כל משאב חישוב קוונטי בפרימיטיב. זה מאפשר לך לכתוב קוד בסגנון פרימיטיב עבור ספקים שעדיין אין להם ממשק מבוסס פרימיטיבים. ניתן להשתמש במחלקות אלה בדיוק כמו Sampler ו-Estimator הרגילים, אלא שיש לאתחלן עם ארגומנט `backend` נוסף לבחירת מחשב הקוונטי להרצה עליו. הם נגישים באמצעות Qiskit. ראה את המדריך [פרימיטיבים של backend](/guides/get-started-with-backend-primitives) לקבלת מידע נוסף.

## אפשרויות
ניתן להעביר אפשרויות לפרימיטיבים כדי להתאים אותם לצרכיך. בעוד שממשק מתודת `run()` של הפרימיטיבים משותף לכל המימושים, האפשרויות שלהם אינן. עיין בעזרי ה-API עבור מימוש פרימיטיב ספציפי כדי ללמוד על האפשרויות שהוא תומך בהן.

לדוגמה, עיין בנושאי [אפשרויות Estimator](/guides/estimator-options) ו[אפשרויות Sampler](/guides/sampler-options) כדי ללמוד על אפשרויות פרימיטיבי Qiskit Runtime, או ראה את [עזרי ה-API של Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) לאפשרויות פרימיטיבי Qiskit Aer.

## יתרונות פרימיטיבי Qiskit
עם פרימיטיבים, משתמשי Qiskit יכולים לכתוב קוד קוונטי עבור QPU ספציפי מבלי לנהל כל פרט באופן מפורש. כמו כן, בשל שכבת ההפשטה הנוספת, ייתכן שתוכל לגשת ביתר קלות ליכולות החומרה המתקדמות של ספק נתון. לדוגמה, עם פרימיטיבי Qiskit Runtime, תוכל לנצל את ההתקדמות האחרונה בהפחתה ודיכוי שגיאות על ידי שינוי אפשרויות כגון [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) של הפרימיטיב, במקום לבנות מימוש משלך של טכניקות אלה.

עבור ספקי חומרה, מימוש פרימיטיבים באופן מקורי מאפשר לך לספק למשתמשים שלך דרך "מהקופסה" יותר לגשת לתכונות החומרה שלך כגון טכניקות עיבוד לאחר מתקדמות. לכן קל יותר למשתמשים שלך ליהנות מהיכולות הטובות ביותר של החומרה שלך.

## הצעדים הבאים
> **Tip:** - הבן את [הקלט והפלט של הפרימיטיב](/guides/primitive-input-output).
> - ראה [דוגמאות מפורטות](/guides/simulate-with-qiskit-sdk-primitives).
> - תרגל עם פרימיטיבים על ידי עבודה דרך [שיעור פונקציית עלות](/learning/courses/variational-algorithm-design/cost-functions) ב-IBM Quantum Learning.
> - ראה [יצירת ספק](/guides/create-a-provider) כדי ללמוד כיצד ליישם פרימיטיבי Sampler ו-Estimator משלך.
> - ראה את [עזרי ה-API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - קרא [העברה לפרימיטיבים V2](/guides/v2-primitives).
> - למד על [פרימיטיבי Qiskit Runtime](/guides/qiskit-runtime-primitives), המשמשים להרצת Circuits על QPU של IBM.